<a href="https://colab.research.google.com/github/bibanaksu/PM25-Forecasting-GRU/blob/main/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import numpy as np

In [25]:
pm = pd.read_csv("pm25.csv")
gee = pd.read_csv("gee_features_daily.csv")
comb = pd.read_csv("combined_gee_pm25.csv")

files_dict = {
    "pm25.csv": pm,
    "gee_features_daily.csv": gee,
    "combined_gee_pm25.csv": comb,
}

for name, df in files_dict.items():
    print("=====", name, "=====")
    print("rows, columns:", df.shape)
    print(df.dtypes)
    print(df.head(3).to_string())
    print()

===== pm25.csv =====
rows, columns: (88041, 5)
date            object
latitude       float64
longitude      float64
pm25           float64
sensor_name     object
dtype: object
         date   latitude  longitude  pm25              sensor_name
0  2016-11-21  38.788056  -0.697222   4.0                ONTINYENT
1  2016-11-21  39.456111  -0.375833   3.0  VALÈNCIA-PISTA DE SILLA
2  2016-11-21  39.480300  -0.336400   2.0      VALÈNCIA-POLITÈCNIC

===== gee_features_daily.csv =====
rows, columns: (164208, 13)
date                    object
datetime_utc            object
id                       int64
lat                    float64
lon                    float64
temperature_celsius    float64
pressure_mb            float64
wind_u                 float64
wind_v                 float64
NO2                    float64
CO                     float64
O3                     float64
AOD                    float64
dtype: object
         date         datetime_utc  id        lat       lon  temperature_ce

In [26]:
rows = []
for name, df in files_dict.items():
    lat_col = "lat" if "lat" in df.columns else "latitude"
    lon_col = "lon" if "lon" in df.columns else "longitude"
    dates = pd.to_datetime(df["date"], errors="coerce")
    rows.append({
        "file": name,
        "n_rows": df.shape[0],
        "n_columns": df.shape[1],
        "date_min": dates.min().date(),
        "date_max": dates.max().date(),
        "lat_min": round(df[lat_col].min(), 3),
        "lat_max": round(df[lat_col].max(), 3),
        "lon_min": round(df[lon_col].min(), 3),
        "lon_max": round(df[lon_col].max(), 3),
    })
description = pd.DataFrame(rows)
print(description.to_string(index=False))

# units of each variable (taken from the project PDF and the column names)
units = {
    "pm25": "ug/m3 (target)",
    "temperature_celsius": "degree Celsius",
    "pressure_mb": "mb (= hPa)",
    "wind_u": "m/s (east-west component)",
    "wind_v": "m/s (north-south component)",
    "NO2": "mol/m2 (Sentinel-5P column, to confirm)",
    "CO": "mol/m2 (Sentinel-5P column, to confirm)",
    "O3": "mol/m2 (Sentinel-5P column, to confirm)",
    "AOD": "dimensionless",
    "latitude / longitude": "decimal degrees",
    "date": "YYYY-MM-DD (daily)",
}
for k, v in units.items():
    print(f"{k:22s} -> {v}")

                  file  n_rows  n_columns   date_min   date_max  lat_min  lat_max  lon_min  lon_max
              pm25.csv   88041          5 2016-11-21 2026-04-16   34.290   43.689   -5.663    9.446
gee_features_daily.csv  164208         13 2016-11-21 2026-04-06   36.133   43.530   -5.663    9.136
 combined_gee_pm25.csv   41718         15 2016-11-25 2026-04-06   36.133   43.530   -5.663    9.136
pm25                   -> ug/m3 (target)
temperature_celsius    -> degree Celsius
pressure_mb            -> mb (= hPa)
wind_u                 -> m/s (east-west component)
wind_v                 -> m/s (north-south component)
NO2                    -> mol/m2 (Sentinel-5P column, to confirm)
CO                     -> mol/m2 (Sentinel-5P column, to confirm)
O3                     -> mol/m2 (Sentinel-5P column, to confirm)
AOD                    -> dimensionless
latitude / longitude   -> decimal degrees
date                   -> YYYY-MM-DD (daily)


In [27]:
missing_tables = {}
for name, df in files_dict.items():
    miss = pd.DataFrame({
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
    })
    missing_tables[name] = miss
    print("=====", name, "=====")
    print(miss[miss["n_missing"] > 0] if miss["n_missing"].sum() > 0 else "no missing values")
    print()

===== pm25.csv =====
no missing values

===== gee_features_daily.csv =====
     n_missing  pct_missing
NO2      19603        11.94
CO       19603        11.94
O3       19914        12.13

===== combined_gee_pm25.csv =====
     n_missing  pct_missing
NO2       2410         5.78
CO        2410         5.78
O3        2451         5.88



In [28]:
dup_rows = []
for name, df in files_dict.items():
    lat_col = "lat" if "lat" in df.columns else "latitude"
    lon_col = "lon" if "lon" in df.columns else "longitude"
    n_full = df.duplicated().sum()
    n_key = df.duplicated(subset=["date", lat_col, lon_col]).sum()
    if "sensor_name" in df.columns:
        n_station = df.duplicated(subset=["date", "sensor_name"]).sum()
        n_station_xy = df.duplicated(subset=["date", "sensor_name", lat_col, lon_col]).sum()
    else:
        n_station = n_station_xy = np.nan
    dup_rows.append({
        "file": name,
        "full_row_duplicates": n_full,
        "dup_date_lat_lon": n_key,
        "dup_date_sensor_name": n_station,
        "dup_date_sensor_lat_lon": n_station_xy,
    })
dup_table = pd.DataFrame(dup_rows)
print(dup_table.to_string(index=False))

# the date+sensor_name duplicates: are they real duplicates or different places?
x = pm[pm.duplicated(subset=["date", "sensor_name"], keep=False)].sort_values(["sensor_name", "date"])
print()
print("rows involved in date+sensor_name repetition:", len(x))
print("sensor names involved:", sorted(x["sensor_name"].unique()))
print(x.head(6).to_string())

                  file  full_row_duplicates  dup_date_lat_lon  dup_date_sensor_name  dup_date_sensor_lat_lon
              pm25.csv                    0                 0                1558.0                      0.0
gee_features_daily.csv                    0                 0                   NaN                      NaN
 combined_gee_pm25.csv                    0                 0                   2.0                      0.0

rows involved in date+sensor_name repetition: 3037
sensor names involved: ['Barcelona', 'LA ALJORRA', 'MOLINA DE SEGURA', 'MOMPEAN', 'SAN BASILIO', 'SSOO']
             date   latitude  longitude  pm25 sensor_name
37798  2023-05-07  41.385315     2.1538   9.0   Barcelona
37799  2023-05-07  41.387490     2.1152   9.0   Barcelona
37877  2023-05-09  41.385315     2.1538  13.0   Barcelona
37878  2023-05-09  41.387490     2.1152   8.0   Barcelona
37916  2023-05-10  41.385315     2.1538   5.0   Barcelona
37917  2023-05-10  41.387490     2.1152   7.0   Barcelona


In [29]:
ts_info = {}
for name, df in files_dict.items():
    d = pd.to_datetime(df["date"], errors="coerce")
    ts_info[name] = {
        "unparsable_dates": int(d.isna().sum()),
        "date_min": d.min().date(),
        "date_max": d.max().date(),
        "is_sorted": bool(d.is_monotonic_increasing),
    }
    if "datetime_utc" in df.columns:
        dt = pd.to_datetime(df["datetime_utc"], errors="coerce")
        ts_info[name]["date_vs_datetime_mismatch"] = int((d != dt.dt.normalize()).sum())
        ts_info[name]["rows_with_hour_not_00"] = int((dt.dt.hour != 0).sum())
print(pd.DataFrame(ts_info).T.to_string())

# rows per year in pm25.csv
pm["date_dt"] = pd.to_datetime(pm["date"])
print()
print(pm["date_dt"].dt.year.value_counts().sort_index())

# last date of the GEE features vs last date of pm25.csv
gee_last = pd.to_datetime(gee["date"]).max()
print()
print("last GEE date:", gee_last.date())
print("pm25 rows after the last GEE date:", int((pm["date_dt"] > gee_last).sum()))

# daily regularity per station (station = name + coordinates)
pm["station_key"] = (pm["sensor_name"].str.strip() + "|" + pm["latitude"].astype(str)
                     + "|" + pm["longitude"].astype(str))
g = pm.groupby("station_key")["date_dt"].agg(["min", "max", "count"])
g["n_days_span"] = (g["max"] - g["min"]).dt.days + 1
g["n_missing_days"] = g["n_days_span"] - g["count"]
g["pct_missing_days"] = (100 * g["n_missing_days"] / g["n_days_span"]).round(1)
print()
print("stations:", len(g))
print(g[["count", "n_days_span", "n_missing_days", "pct_missing_days"]].describe().round(1).to_string())

                       unparsable_dates    date_min    date_max is_sorted date_vs_datetime_mismatch rows_with_hour_not_00
pm25.csv                              0  2016-11-21  2026-04-16      True                       NaN                   NaN
gee_features_daily.csv                0  2016-11-21  2026-04-06      True                         0                     0
combined_gee_pm25.csv                 0  2016-11-25  2026-04-06      True                         0                     0

date_dt
2016      312
2017     3182
2018     2563
2019     6278
2020     5618
2021     8021
2022     9489
2023    10927
2024    19101
2025    18007
2026     4543
Name: count, dtype: int64

last GEE date: 2026-04-06
pm25 rows after the last GEE date: 337

stations: 133
        count  n_days_span  n_missing_days  pct_missing_days
count   133.0        133.0           133.0             133.0
mean    662.0        888.5           226.5              17.1
std     431.4        708.1           393.3              19.

In [33]:

# =====================================================
# CELL 8 - STATION IDs AND SENSOR IDENTITY
# =====================================================

print("========== PM2.5 DATASET: STATION INFORMATION ==========")

# Create a normalized sensor name (remove spaces at beginning/end)
pm["sensor_name_clean"] = pm["sensor_name"].str.strip()

# Basic station information
print("Unique original sensor names:", pm["sensor_name"].nunique())
print("Unique sensor names after cleaning:", pm["sensor_name_clean"].nunique())

print(
    "Unique coordinate pairs:",
    pm[["latitude", "longitude"]].drop_duplicates().shape[0]
)

print("Missing sensor names:", pm["sensor_name"].isna().sum())


# -----------------------------------------------------
# 1. FIND SENSOR NAMES WITH EXTRA SPACES
# -----------------------------------------------------

space_names = pm.loc[
    pm["sensor_name"].notna() &
    (pm["sensor_name"] != pm["sensor_name"].str.strip()),
    "sensor_name"
].unique()

print("\nNames with extra spaces:", list(space_names))


# -----------------------------------------------------
# 2. CHECK IF ONE SENSOR NAME HAS SEVERAL COORDINATES
# -----------------------------------------------------

multi_xy = pm.groupby("sensor_name_clean")[
    ["latitude", "longitude"]
].nunique()

multi_xy = multi_xy[
    (multi_xy["latitude"] > 1) |
    (multi_xy["longitude"] > 1)
]

print("\nSensor names with more than one coordinate pair:",
      len(multi_xy))

print(multi_xy.to_string())


# -----------------------------------------------------
# 3. CHECK IF ONE COORDINATE PAIR HAS SEVERAL NAMES
# -----------------------------------------------------

multi_name = pm.groupby(
    ["latitude", "longitude"]
)["sensor_name_clean"].nunique()

shared_coordinates = multi_name[multi_name > 1]

print("\nCoordinate pairs associated with multiple sensor names:",
      len(shared_coordinates))

print(shared_coordinates.to_string())


# -----------------------------------------------------
# 4. GEE FEATURES: CHECK UNIQUE IDS
# -----------------------------------------------------

print("\n========== GEE FEATURES: STATION INFORMATION ==========")

print(
    "gee_features_daily: unique IDs =",
    gee["id"].nunique()
)

print(
    "combined_gee_pm25: unique IDs =",
    comb["id"].nunique()
)

print(
    "combined_gee_pm25: unique sensor names =",
    comb["sensor_name"].nunique()
)


# -----------------------------------------------------
# 5. CHECK WHETHER ONE GEE ID HAS SEVERAL SENSOR NAMES
# -----------------------------------------------------

comb_name_check = comb.groupby("id")["sensor_name"].nunique()

ids_with_multiple_names = comb_name_check[
    comb_name_check > 1
]

print(
    "\nCombined dataset: IDs associated with multiple sensor names =",
    len(ids_with_multiple_names)
)

print(ids_with_multiple_names.to_string())


# -----------------------------------------------------
# 6. CHECK WHETHER ONE SENSOR NAME HAS SEVERAL GEE IDS
# -----------------------------------------------------

comb_id_check = comb.groupby("sensor_name")["id"].nunique()

names_with_multiple_ids = comb_id_check[
    comb_id_check > 1
]

print(
    "\nCombined dataset: sensor names associated with multiple IDs =",
    len(names_with_multiple_ids)
)

print(names_with_multiple_ids.to_string())


# -----------------------------------------------------
# 7. CHECK COORDINATE CONSISTENCY FOR GEE IDS
# -----------------------------------------------------

gee_coord_check = gee.groupby("id")[["lat", "lon"]].nunique()

gee_inconsistent_ids = gee_coord_check[
    (gee_coord_check["lat"] > 1) |
    (gee_coord_check["lon"] > 1)
]

print(
    "\nGEE features: IDs with inconsistent coordinates =",
    len(gee_inconsistent_ids)
)

comb_coord_check = comb.groupby("id")[
    ["latitude", "longitude"]
].nunique()

comb_inconsistent_ids = comb_coord_check[
    (comb_coord_check["latitude"] > 1) |
    (comb_coord_check["longitude"] > 1)
]

print(
    "Combined dataset: IDs with inconsistent coordinates =",
    len(comb_inconsistent_ids)
)





========== PM2.5 DATASET: STATION INFORMATION ==========
Unique original sensor names: 125
Unique sensor names after cleaning: 125
Unique coordinate pairs: 133
Missing sensor names: 0

Names with extra spaces: ['Estación de Franca ', 'Viladecavalls ']

Sensor names with more than one coordinate pair: 7
                   latitude  longitude
sensor_name_clean                     
Barcelona                 3          3
LA ALJORRA                2          2
MOLINA DE SEGURA          2          2
MOMPEAN                   2          2
SAN BASILIO               2          2
SSOO                      2          2
Viladecavalls             2          2

Coordinate pairs associated with multiple sensor names: 0
Series([], )

========== GEE FEATURES: STATION INFORMATION ==========
gee_features_daily: unique IDs = 100
combined_gee_pm25: unique IDs = 98
combined_gee_pm25: unique sensor names = 94

Combined dataset: IDs associated with multiple sensor names = 0
Series([], )

Combined dataset: sen

In [32]:

coord_rows = []

for name, df in files_dict.items():
    lat_col = "lat" if "lat" in df.columns else "latitude"
    lon_col = "lon" if "lon" in df.columns else "longitude"

    coord_rows.append({
        "file": name,
        "rows_with_missing_coordinates": int(
            df[[lat_col, lon_col]].isna().any(axis=1).sum()
        ),
        "lat_out_of_range": int(
            ((df[lat_col] < -90) | (df[lat_col] > 90)).sum()
        ),
        "lon_out_of_range": int(
            ((df[lon_col] < -180) | (df[lon_col] > 180)).sum()
        ),
        "unique_coordinate_pairs": df[
            [lat_col, lon_col]
        ].drop_duplicates().shape[0],
        "latitude_range": (
            round(df[lat_col].min(), 2),
            round(df[lat_col].max(), 2)
        ),
        "longitude_range": (
            round(df[lon_col].min(), 2),
            round(df[lon_col].max(), 2)
        )
    })

coord_table = pd.DataFrame(coord_rows)

print(coord_table.to_string(index=False))


                  file  rows_with_missing_coordinates  lat_out_of_range  lon_out_of_range  unique_coordinate_pairs latitude_range longitude_range
              pm25.csv                              0                 0                 0                      133 (34.29, 43.69)   (-5.66, 9.45)
gee_features_daily.csv                              0                 0                 0                      100 (36.13, 43.53)   (-5.66, 9.14)
 combined_gee_pm25.csv                              0                 0                 0                       98 (36.13, 43.53)   (-5.66, 9.14)


In [35]:
for name, df in [("pm25.csv", pm), ("combined_gee_pm25.csv", comb)]:
    p = df["pm25"]
    print("=====", name, "=====")
    print("missing pm25     :", int(p.isna().sum()))
    print("pm25 <= 0        :", int((p <= 0).sum()),
          "(zeros:", int((p == 0).sum()), "| -999:", int((p == -999).sum()),
          "| other negatives:", int(((p < 0) & (p != -999)).sum()), ")")
    print("pm25 >= 500      :", int((p >= 500).sum()))
    print("min / max        :", p.min(), "/", p.max())
    print()

===== pm25.csv =====
missing pm25     : 0
pm25 <= 0        : 2093 (zeros: 1428 | -999: 96 | other negatives: 569 )
pm25 >= 500      : 19
min / max        : -999.0 / 1000.0

===== combined_gee_pm25.csv =====
missing pm25     : 0
pm25 <= 0        : 727 (zeros: 454 | -999: 4 | other negatives: 269 )
pm25 >= 500      : 8
min / max        : -999.0 / 1000.0



In [36]:
cleaning_log = []

def clean_pm25(df, name):
    df = df.copy()
    n0 = len(df)
    # step 1: remove extra spaces in station names
    df["sensor_name"] = df["sensor_name"].str.strip()
    # step 2: missing PM2.5
    mask_na = df["pm25"].isna()
    n_na = int(mask_na.sum())
    df = df[~mask_na]
    # step 3: invalid sentinel values (<= 0 or >= 500)
    mask_low = df["pm25"] <= 0
    mask_high = df["pm25"] >= 500
    n_low = int(mask_low.sum())
    n_high = int(mask_high.sum())
    df = df[~(mask_low | mask_high)]
    n1 = len(df)
    cleaning_log.append({
        "file": name, "rows_before": n0, "removed_missing_pm25": n_na,
        "removed_pm25_le_0": n_low, "removed_pm25_ge_500": n_high,
        "removed_total": n0 - n1, "rows_after": n1,
        "pct_removed": round(100 * (n0 - n1) / n0, 2),
    })
    return df.reset_index(drop=True)

pm_clean = clean_pm25(pm[["date", "latitude", "longitude", "pm25", "sensor_name"]], "pm25.csv")
comb_clean = clean_pm25(comb, "combined_gee_pm25.csv")

log_df = pd.DataFrame(cleaning_log)
print(log_df.T.to_string())

# duplicates: none found in cell 6 for the full rows and for the key
# (date, sensor, lat, lon), so no row is dropped for duplicates.
print()
print("full-row duplicates left (pm25):", int(pm_clean.duplicated().sum()))
print("key duplicates left (pm25):", int(pm_clean.duplicated(subset=["date", "sensor_name", "latitude", "longitude"]).sum()))

                             0                      1
file                  pm25.csv  combined_gee_pm25.csv
rows_before              88041                  41718
removed_missing_pm25         0                      0
removed_pm25_le_0         2093                    727
removed_pm25_ge_500         19                      8
removed_total             2112                    735
rows_after               85929                  40983
pct_removed                2.4                   1.76

full-row duplicates left (pm25): 0
key duplicates left (pm25): 0


In [38]:

for name, df in [
    ("pm25_clean", pm_clean),
    ("combined_clean", comb_clean)
]:
    print("=====", name, "=====")
    print("Shape:", df.shape)
    print("Missing PM2.5:", int(df["pm25"].isna().sum()))
    print("PM2.5 min / max:", df["pm25"].min(), "/", df["pm25"].max())
    print("Date range:", df["date"].min(), "->", df["date"].max())
    print("Unique sensor names:", df["sensor_name"].nunique())
    print("PM2.5 statistics:")
    print(df["pm25"].describe().round(2).to_string())
    print("Full-row duplicates:", int(df.duplicated().sum()))
    print()

lost = (
    set(pm["sensor_name"].dropna().str.strip())
    - set(pm_clean["sensor_name"].dropna().str.strip())
)

print("Stations with no valid row left after cleaning:", len(lost))
print("Station names:", sorted(lost))


===== pm25_clean =====
Shape: (85929, 5)
Missing PM2.5: 0
PM2.5 min / max: 0.0066666666666666 / 484.9815
Date range: 2016-11-21 -> 2026-04-16
Unique sensor names: 118
PM2.5 statistics:
count    85929.00
mean         9.66
std          8.81
min          0.01
25%          4.86
50%          8.04
75%         12.54
max        484.98
Full-row duplicates: 0

===== combined_clean =====
Shape: (40983, 15)
Missing PM2.5: 0
PM2.5 min / max: 0.0066666666666666 / 484.9815
Date range: 2016-11-25 -> 2026-04-06
Unique sensor names: 91
PM2.5 statistics:
count    40983.00
mean         9.46
std          9.73
min          0.01
25%          4.67
50%          7.85
75%         12.09
max        484.98
Full-row duplicates: 0

Stations with no valid row left after cleaning: 7
Station names: ['064F', 'balcon SE pas de vent', 'balcon SE à 150m', 'grasse', 'mimiz', "réunion capt'air citoyen", 'testcompnex']


In [39]:
pm_clean.to_csv("pm25_clean.csv", index=False, encoding="utf-8")
comb_clean.to_csv("combined_gee_pm25_clean.csv", index=False, encoding="utf-8")
log_df.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
print("files saved")

files saved


In [40]:
lines = []
L = lines.append
a = log_df.set_index("file")

L("CLEANING REPORT - PM2.5 FORECASTING MINI PROJECT")
L("Part: dataset audit and cleaning")
L("=" * 60)
L("")
L("1. DATASET DESCRIPTION")
L("-" * 60)
L("Source  : Kaggle dataset 'Air Quality - Mediterranean Basin (2016-2026)'")
L("          (link given in the project PDF, author: sarahameziane).")
L("          PM2.5 comes from air-quality monitoring stations (pm25.csv);")
L("          meteorological, AOD, NO2, CO and O3 variables come from Google")
L("          Earth Engine (GEE) files.")
L("License : not written in the CSV files. To be copied from the 'License'")
L("          field of the Kaggle page before the final report.")
L("Target  : pm25 = daily surface PM2.5 concentration, in ug/m3.")
L("")
L("Files received:")
for _, r in description.iterrows():
    L(f"  - {r['file']}: {r['n_rows']} rows x {r['n_columns']} columns, "
      f"dates {r['date_min']} -> {r['date_max']}, "
      f"lat {r['lat_min']} to {r['lat_max']}, lon {r['lon_min']} to {r['lon_max']}")
L("")
L("Units:")
for k, v in units.items():
    L(f"  - {k}: {v}")
L("")
L(f"Geographical coverage: western Mediterranean basin (mainly Spain), latitude "
  f"{pm['latitude'].min():.2f} to {pm['latitude'].max():.2f}, longitude "
  f"{pm['longitude'].min():.2f} to {pm['longitude'].max():.2f}.")
L(f"Time coverage: {pm['date'].min()} to {pm['date'].max()} (daily data) in pm25.csv; "
  f"GEE features stop on {gee_last.date()}.")
L("")
L("2. MISSING VALUES")
L("-" * 60)
for name, miss in missing_tables.items():
    m = miss[miss["n_missing"] > 0]
    if len(m) == 0:
        L(f"  - {name}: no missing value in any column (pm25 included).")
    else:
        txt = ", ".join(f"{i} = {int(r.n_missing)} ({r.pct_missing}%)" for i, r in m.iterrows())
        L(f"  - {name}: {txt}. No missing value in the other columns.")
L("  Only NO2, CO and O3 (Sentinel-5P) have missing values. They are NOT filled here:")
L("  imputation must be done inside each training fold (project PDF, section 5.3.1).")
L("")
L("3. DUPLICATE RECORDS")
L("-" * 60)
for _, r in dup_table.iterrows():
    L(f"  - {r['file']}: full-row duplicates = {r['full_row_duplicates']}, "
      f"(date, lat, lon) duplicates = {r['dup_date_lat_lon']}")
L(f"  In pm25.csv, {int(dup_table.loc[0, 'dup_date_sensor_name'])} rows repeat the same date and "
  "sensor_name, but with different coordinates (6 names: Barcelona, LA ALJORRA,")
L("  MOLINA DE SEGURA, MOMPEAN, SAN BASILIO, SSOO). They are different measuring")
L("  points with the same name, not duplicates, so they are kept.")
L("  Result: no duplicate row to remove.")
L("")
L("4. TIMESTAMPS")
L("-" * 60)
L("  - All dates can be parsed (0 invalid date) and all files are sorted by date.")
L("  - In the GEE files, datetime_utc always equals date at 00:00:00 (0 mismatch),")
L("    so the data are daily.")
L(f"  - pm25.csv has {int((pm['date_dt'] > gee_last).sum())} rows after the last GEE date "
  f"({gee_last.date()}): they have no features.")
L(f"  - Daily regularity: {len(g)} stations (name + coordinates). Median span of "
  f"{g['n_days_span'].median():.0f} days with median {g['pct_missing_days'].median():.1f}% missing days;")
L(f"    the number of days per station goes from {int(g['count'].min())} to {int(g['count'].max())}.")
L("    The series are irregular and do not start/end on the same dates. No gap is")
L("    filled at this stage (no interpolation).")
L("")
L("5. STATION IDs")
L("-" * 60)
L(f"  - pm25.csv has {pm['sensor_name'].nunique()} sensor names and "
  f"{pm[['latitude','longitude']].drop_duplicates().shape[0]} different coordinate pairs; no missing name.")
L("  - 7 names have more than one coordinate pair (the 6 above + Viladecavalls).")
L("    Rule used by the group: a station = sensor_name + latitude + longitude.")
L("  - No coordinate pair is shared by two different names.")
L("  - 2 names had a space at the end ('Estación de Franca ', 'Viladecavalls '):")
L("    spaces removed.")
L(f"  - GEE files use a numeric id: {gee['id'].nunique()} ids in gee_features_daily.csv, "
  f"{comb['id'].nunique()} ids and {comb['sensor_name'].nunique()} names in combined_gee_pm25.csv.")
L("    Each id has one single coordinate pair and one single name; 4 names (LA ALJORRA,")
L("    MOLINA DE SEGURA, MOMPEAN, SAN BASILIO) have 2 ids, which is consistent with")
L("    their 2 coordinate pairs.")
L("")
L("6. COORDINATES")
L("-" * 60)
L("  - No missing latitude/longitude, none outside [-90, 90] / [-180, 180] in any file.")
L("  - All points are inside the western Mediterranean area given above.")
L(f"  - {pm[['latitude','longitude']].drop_duplicates().shape[0]} distinct points in pm25.csv, "
  f"{gee[['lat','lon']].drop_duplicates().shape[0]} in gee_features_daily.csv, "
  f"{comb[['latitude','longitude']].drop_duplicates().shape[0]} in combined_gee_pm25.csv.")
L("  - The GEE ids are fixed to one point each, so coordinates are reliable for")
L("    spatial CV. Matching pm25 stations to GEE ids (about 0.25 degree) is a later step.")
L("")
L("7. INVALID PM2.5 VALUES (before cleaning)")
L("-" * 60)
for name, df in [("pm25.csv", pm), ("combined_gee_pm25.csv", comb)]:
    p = df["pm25"]
    L(f"  - {name}: missing = {int(p.isna().sum())}, pm25 <= 0 = {int((p <= 0).sum())} "
      f"(zeros {int((p == 0).sum())}, -999 {int((p == -999).sum())}, other negatives "
      f"{int(((p < 0) & (p != -999)).sum())}), pm25 >= 500 = {int((p >= 500).sum())}, "
      f"min = {p.min()}, max = {p.max()}")
L("")
L("8. CLEANING STEPS")
L("-" * 60)
L("  Step 1: remove the spaces at the beginning/end of sensor_name.")
L("  Step 2: remove rows with missing PM2.5 (none found).")
L("  Step 3: remove rows with PM2.5 <= 0 (sentinel -999, zeros and negative values,")
L("          which are not valid concentrations).")
L("  Step 4: remove rows with PM2.5 >= 500 (invalid sentinel values, rule of the PDF).")
L("  Step 5: duplicates, timestamps and coordinates: checked, nothing to remove.")
L("  Nothing else is changed: valid extreme PM2.5 values below 500 are kept (they can")
L("  be real pollution episodes), NO2/CO/O3 are not imputed, no outlier removal,")
L("  no scaling, no lag creation.")
L("  The rules are applied before EDA and before any split, as asked in section 5.1.")
L("")
L("  Counts:")
for f, r in a.iterrows():
    L(f"  - {f}: {int(r['rows_before'])} rows before; removed missing = {int(r['removed_missing_pm25'])}, "
      f"<= 0 = {int(r['removed_pm25_le_0'])}, >= 500 = {int(r['removed_pm25_ge_500'])}; "
      f"total removed = {int(r['removed_total'])} ({r['pct_removed']}%); {int(r['rows_after'])} rows after.")
L("")
L("9. RESULT AFTER CLEANING")
L("-" * 60)
for name, df in [("pm25_clean.csv", pm_clean), ("combined_gee_pm25_clean.csv", comb_clean)]:
    L(f"  - {name}: {df.shape[0]} rows x {df.shape[1]} columns, {df['sensor_name'].nunique()} sensor names, "
      f"dates {df['date'].min()} -> {df['date'].max()}, pm25 min = {df['pm25'].min():.3f}, "
      f"max = {df['pm25'].max():.2f}, mean = {df['pm25'].mean():.2f}, median = {df['pm25'].median():.2f}.")
L(f"  - {len(lost)} station names in pm25.csv have no valid row left after cleaning.")
L("  - Checks after cleaning: no missing PM2.5, no value <= 0 or >= 500, no duplicate.")
L("")
L("10. FILES SHARED WITH THE GROUP")
L("-" * 60)
L("  - pm25_clean.csv")
L("  - combined_gee_pm25_clean.csv")
L("  - cleaning_log.csv (row counts of each step)")
L("  - cleaning_report.txt (this file)")
L("  gee_features_daily.csv is unchanged (it has no PM2.5 column).")
L("")

with open("cleaning_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("\n".join(lines))

CLEANING REPORT - PM2.5 FORECASTING MINI PROJECT
Part: dataset audit and cleaning

1. DATASET DESCRIPTION
------------------------------------------------------------
Source  : Kaggle dataset 'Air Quality - Mediterranean Basin (2016-2026)'
          (link given in the project PDF, author: sarahameziane).
          PM2.5 comes from air-quality monitoring stations (pm25.csv);
          meteorological, AOD, NO2, CO and O3 variables come from Google
          Earth Engine (GEE) files.
License : not written in the CSV files. To be copied from the 'License'
          field of the Kaggle page before the final report.
Target  : pm25 = daily surface PM2.5 concentration, in ug/m3.

Files received:
  - pm25.csv: 88041 rows x 5 columns, dates 2016-11-21 -> 2026-04-16, lat 34.29 to 43.689, lon -5.663 to 9.446
  - gee_features_daily.csv: 164208 rows x 13 columns, dates 2016-11-21 -> 2026-04-06, lat 36.133 to 43.53, lon -5.663 to 9.136
  - combined_gee_pm25.csv: 41718 rows x 15 columns, dates 2016-11